In [1]:
# ============================================================================
# GEE DATA DOWNLOAD FOR ML INFERENCE - MASTER CONTROL NOTEBOOK
# ============================================================================
#
# Master Control Notebook for Google Earth Engine Data Processing
# This notebook controls the complete data download and preprocessing pipeline
# for ML model inference. It replicates your original preprocessing workflow
# using Google Earth Engine in Google Colab.
#
# Features:
# - S2 Harmonization: Exact replication of harmonize.py offset correction
# - GHSL Processing: Same threshold and processing as create_ref.py
# - ML Compatibility: Perfect format matching for trained models
# - Spatial Alignment: Ensures all datasets are properly aligned
# - Validation: Comprehensive checks for data quality
#
# Requirements:
# - Google Earth Engine account with access
# - Google Colab environment
# - All scripts uploaded to Colab (gee_auth.py, download_s2.py, etc.)

# ============================================================================
# CELL 1: Setup and Installation
# ============================================================================

# Install required packages
!pip install earthengine-api geemap geopandas rioxarray rasterstats folium -q

# Import standard libraries
import os
import sys
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive (optional, for saving results)
from google.colab import drive
drive.mount('/content/drive')

print("✅ Setup completed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.2/62.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.3 MB/s eta 0:00:00


ValueError: mount failed

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.downloads.gee_auth import initialize_gee, check_gee_status
from src.downloads.download_s2 import S2Downloader, quick_download_s2
from src.downloads.download_ghsl import GHSLDownloader, quick_download_ghsl, create_reference_labels_quick
from src.downloads.download_utils import create_aoi_from_bounds, validate_aoi, download_ee_image, check_files_alignment, get_file_info, create_aoi_from_file
from src.downloads.data_validator import MLCompatibilityValidator, quick_validate_dataset, check_ml_compatibility

In [3]:

# ============================================================================
# CELL 3: Configuration Parameters
# ============================================================================

# ============================================================================
# MAIN CONFIGURATION - EDIT THESE PARAMETERS FOR YOUR PROJECT
# ============================================================================

# 🗺️ GEOGRAPHIC CONFIGURATION
# Define your Area of Interest (AOI)
# Option 1: Bounding box (minx, miny, maxx, maxy) in WGS84 coordinates
#AOI_BOUNDS = (106.7, -6.4, 107.0, -6.1)  # Example: Jakarta area

# Option 2: Path to AOI file (uncomment and modify if using file)
AOI_GEOJSON_PATH = "/content/drive/MyDrive/Fine-Tuning-Workflow/Data/AMBA_IDE_AOI.geojson"
#AOI_GEOJSON_PATH = None

# 📅 TEMPORAL CONFIGURATION
START_DATE = "2020-06-01"  # Start date for satellite data
END_DATE = "2020-8-31"    # End date for satellite data
CLOUD_FILTER = 10          # Maximum cloud percentage for S2

# 📁 OUTPUT CONFIGURATION
OUTPUT_DIR = "/content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/S2"  # Local output directory
SAVE_TO_DRIVE = True                    # Also save to Google Drive
DRIVE_FOLDER = "ML_Inference_Data"      # Google Drive folder name

# 🛰️ SENTINEL-2 CONFIGURATION (CRITICAL FOR ML COMPATIBILITY)
S2_BANDS = ['B2', 'B3', 'B4','B5', 'B6','B7','B8', 'B8A','B11', 'B12']  # Same as original pipeline
S2_SCALE = 10                            # Resolution in meters
S2_APPLY_HARMONIZATION = False            # Apply offset correction (harmonize.py equivalent)
S2_NORMALIZE = False                      # Scale to [0,1] range for ML
S2_COMPOSITE_METHOD = "median"           # Temporal reduction method
S2_FILENAME = "s2_2020_AMBA.tif"

# 🏗️ GHSL CONFIGURATION (MUST MATCH create_ref.py)
GHSL_YEAR = 2020                         # GHSL data year
GHSL_THRESHOLD = 15                      # Built-up threshold (SAME AS create_ref.py)
GHSL_RESAMPLE_TO_S2 = True              # Resample to match S2 grid
GHSL_FILENAME = "ghsl_builtup_partidos_amba.tif"

# 🏷️ REFERENCE LABELS CONFIGURATION
CREATE_REFERENCE_LABELS = True           # Create reference labels
REFERENCE_FILENAME = "reference_labels_partidos_AMBA.tif"
DUAS_GEOJSON_PATH = "/content/drive/MyDrive/Fine-Tuning-Workflow/Data/renabap-datos-barrios.geojson"  # Uncomment if you have DUA polygons
#DUAS_GEOJSON_PATH = None

# 🔧 PROCESSING CONFIGURATION
MAX_PIXELS = 1e9                         # Maximum pixels for download
DOWNLOAD_METHOD = "drive"               # "direct" or "drive" export
VALIDATE_OUTPUTS = True                  # Run validation checks
CREATE_PATCHES = False                   # Create inference patches (set True if needed)
PATCH_SIZE = 128                         # Patch size for ML inference

# 🌍 GEE CONFIGURATION
GEE_PROJECT_ID = "ee-techo"                    # Your GEE project ID (optional)

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"📁 Output directory: {OUTPUT_DIR}")
print(f"🗺️ AOI: {AOI_BOUNDS if AOI_GEOJSON_PATH is None else AOI_GEOJSON_PATH}")
print(f"📅 Date range: {START_DATE} to {END_DATE}")
print(f"🛰️ S2 harmonization: {'ON' if S2_APPLY_HARMONIZATION else 'OFF'}")
print(f"🏗️ GHSL threshold: >{GHSL_THRESHOLD} (matching create_ref.py)")

📁 Output directory: /content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/S2
🗺️ AOI: /content/drive/MyDrive/Fine-Tuning-Workflow/Data/AMBA_IDE_AOI.geojson
📅 Date range: 2020-06-01 to 2020-8-31
🛰️ S2 harmonization: OFF
🏗️ GHSL threshold: >15 (matching create_ref.py)


In [4]:
# ============================================================================
# CELL 4: Initialize Google Earth Engine
# ============================================================================

# Initialize Google Earth Engine
print("🌍 Initializing Google Earth Engine...")

success = initialize_gee(project_id=GEE_PROJECT_ID)

if success:
    # Check status
    status = check_gee_status()
    print("\n📊 GEE Status:")
    for key, value in status.items():
        print(f"   {key}: {value}")
else:
    print("❌ GEE initialization failed. Please check authentication.")
    print("Try running: ee.Authenticate() manually if needed.")

🌍 Initializing Google Earth Engine...
⚙️ Setting up Colab environment optimizations...
🚀 GPU detected - configuring for GPU environment
✅ Colab environment configured
🌍 Initializing Google Earth Engine...
🔐 Authentication attempt 1/3
👤 Attempting user authentication...
📝 Please follow the authentication prompts below:
✅ User authentication successful
🚀 Initializing with project ID: ee-techo
✅ Google Earth Engine initialized successfully!

🌍 GOOGLE EARTH ENGINE STATUS
✅ Authentication: SUCCESS
✅ Initialization: SUCCESS
🔑 Auth Method: user
🏗️ Project ID: ee-techo


📊 GEE Status:
   authenticated: True
   initialized: True
   auth_method: unknown
   project_id: None
   can_access_gee: True
   error: None


In [5]:
# ============================================================================
# CELL 5: Create and Validate AOI
# ============================================================================

# Create AOI geometry
print("🗺️ Creating AOI geometry...")

if AOI_GEOJSON_PATH:
    aoi_geom = create_aoi_from_file(AOI_GEOJSON_PATH)
    aoi_input = AOI_GEOJSON_PATH
else:
    aoi_geom = create_aoi_from_bounds(AOI_BOUNDS)
    aoi_input = AOI_BOUNDS

# Validate AOI
validation = validate_aoi(aoi_input)

if validation['valid']:
    print("✅ AOI validation passed:")
    print(f"   Area: {validation['area_km2']:.1f} km²")
    print(f"   Bounds: {validation['bounds']}")

    if validation['warnings']:
        print("   ⚠️ Warnings:")
        for warning in validation['warnings']:
            print(f"     - {warning}")
else:
    print(f"❌ AOI validation failed: {validation.get('error', 'Unknown error')}")
    print("Please check your AOI configuration.")

🗺️ Creating AOI geometry...
✅ AOI validation passed:
   Area: 4841.9 km²
   Bounds: {'minx': -59.0562718, 'miny': -35.011275000000026, 'maxx': -58.01768779, 'maxy': -34.00189208999998}


In [ ]:
aoi

In [6]:
# ============================================================================
# CELL 6: Download Sentinel-2 Data
# ============================================================================

print("📡 Starting Sentinel-2 download...")
print("="*50)

try:
    # Initialize S2 downloader
    s2_downloader = S2Downloader(validate_outputs=VALIDATE_OUTPUTS)

    # Download S2 data with harmonization
    s2_filepath = s2_downloader.download_s2_harmonized(
        tile_size_deg=0.05,
        aoi_geom=aoi_geom,
        start_date=START_DATE,
        end_date=END_DATE,
        bands=S2_BANDS,
        cloud_filter=CLOUD_FILTER,
        scale=S2_SCALE,
        apply_harmonization=S2_APPLY_HARMONIZATION,
        normalize=S2_NORMALIZE,
        composite_method=S2_COMPOSITE_METHOD,
        output_filename=os.path.join(OUTPUT_DIR, S2_FILENAME)
    )

    # Get download statistics
    s2_stats = s2_downloader.get_download_stats()

    print("\n📊 S2 Download Statistics:")
    for key, value in s2_stats.items():
        print(f"   {key}: {value}")

    # Show file info
    s2_info = get_file_info(s2_filepath)
    print("\n📋 S2 File Information:")
    for key, value in s2_info.items():
        if key != 'statistics':  # Skip detailed stats for cleaner output
            print(f"   {key}: {value}")

    print(f"\n✅ Sentinel-2 download completed: {s2_filepath}")

except Exception as e:
    print(f"❌ Sentinel-2 download failed: {str(e)}")
    print("\nTroubleshooting tips:")
    print("- Check if AOI is too large (try smaller area)")
    print("- Verify date range has available images")
    print("- Consider using Drive export for large areas")
    s2_filepath = None


📡 Starting Sentinel-2 download...
📡 Downloading Sentinel-2 data...
   📅 Date range: 2020-06-01 to 2020-8-31
   📊 Bands: ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
   ☁️ Cloud filter: 10%
   🔧 Harmonization: OFF
   📏 Scale: 10m
📊 Found 26 Sentinel-2 images
🔄 Creating median composite...
📊 Estimated download size: 4927.2 MB
📐 AOI dimensions: 115.3 x 112.0 km
🔄 Area too large for direct download - using tiled approach...
🗂️ Creating tiles (size: 0.05° = ~5.6 km)...


📦 Created 242 tiles for download
📥 Downloading tile 1/242...
🌐 Requesting download from GEE...
📦 Processing download...
📥 Downloading tile 2/242...
🌐 Requesting download from GEE...
📦 Processing download...
📥 Downloading tile 3/242...
🌐 Requesting download from GEE...
📦 Processing download...
📥 Downloading tile 4/242...
🌐 Requesting download from GEE...
📦 Processing download...
📥 Downloading tile 5/242...
🌐 Requesting download from GEE...
📦 Processing download...
📥 Downloading tile 6/242...
🌐 Requesting download from GEE...
📦 Processing download...
📥 Downloading tile 7/242...
🌐 Requesting download from GEE...
📦 Processing download...
📥 Downloading tile 8/242...
🌐 Requesting download from GEE...
📦 Processing download...
📥 Downloading tile 9/242...
🌐 Requesting download from GEE...
📦 Processing download...
📥 Downloading tile 10/242...
🌐 Requesting download from GEE...
📦 Processing download...
📥 Downloading tile 11/242...
🌐 Requesting download from GEE...
📦 Processing download...
📥 Downlo

✅ Downloaded 240/242 tiles successfully
🔗 Mosaicking tiles...
🔗 Mosaicking 240 tiles...


📊 Mosaicking 240 valid tiles...
✅ Mosaic saved: /content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/S2/s2_2020_AMBA.tif
📊 Final mosaic size: 1308.4 MB
🧹 Cleaning up temporary files...
🔍 Validating S2 output format...
⚠️ Warning: Data range [1.000, 17568.000] outside expected range [0, 10000]
✅ Validation passed:
   📊 Bands: 10
   📏 Shape: (11237, 11563)
   🔢 Data type: float32
   📈 Data range: [1.000, 17568.000]
   🗺️ CRS: EPSG:4326
⚠️ Warning: Could not update statistics: 'ImageCollection' object has no attribute 'aggregate_min_max'
✅ S2 data downloaded successfully: /content/drive/MyDrive/Fine-Tuning-Workflow/Downloads/S2/s2_2020_AMBA.tif

📊 S2 Download Statistics:

📋 S2 File Information:
   filename: s2_2020_AMBA.tif
   shape: (11237, 11563)
   bands: 10
   dtype: float32
   crs: EPSG:4326
   transform: [8.983152841195215e-05, 0.0, -59.056324756358286, 0.0, -8.983152841195215e-05, -34.00186232462277, 0.0, 0.0, 1.0]
   bounds: {'minx': -59.056324756358286, 'miny': -35.0112992093878

In [ ]:
# ============================================================================
# CELL 7 & 8 COMBINED: Download GHSL and Create Reference Labels (No Redundancy)
# ============================================================================

print("🏗️ Starting GHSL download and reference creation...")
print("="*50)

ghsl_filepath = None
reference_filepath = None

try:
    # ========================================================================
    # STEP 1: Download GHSL Binary Mask
    # ========================================================================
    print("\n📡 STEP 1: Downloading GHSL built-up data...")
    print("-"*50)

    # Initialize GHSL downloader
    ghsl_downloader = GHSLDownloader(validate_outputs=VALIDATE_OUTPUTS)

    # Download GHSL data
    ghsl_filepath = ghsl_downloader.download_ghsl_builtup(
        aoi_geom=aoi_geom,
        year=GHSL_YEAR,
        threshold=GHSL_THRESHOLD,
        resample_to_s2=GHSL_RESAMPLE_TO_S2,
        target_scale=S2_SCALE,
        output_filename=os.path.join(OUTPUT_DIR, GHSL_FILENAME)
    )

    # Get download statistics
    ghsl_stats = ghsl_downloader.get_download_stats()
    print("\n📊 GHSL Download Statistics:")
    for key, value in ghsl_stats.items():
        print(f"   {key}: {value}")

    # Show file info
    ghsl_info = get_file_info(ghsl_filepath)
    print("\n📋 GHSL File Information:")
    for key, value in ghsl_info.items():
        print(f"   {key}: {value}")

    print(f"\n✅ GHSL download completed: {ghsl_filepath}")

    # ========================================================================
    # STEP 2: Create Reference Labels (Using Local GHSL File)
    # ========================================================================
    if CREATE_REFERENCE_LABELS:
        print("\n🏷️ STEP 2: Creating reference labels from local GHSL file...")
        print("-"*50)

        # Load DUA polygons if provided
        if DUAS_GEOJSON_PATH:
            print(f"📍 Including DUA polygons from: {DUAS_GEOJSON_PATH}")
            import geopandas as gpd
            duas_gdf = gpd.read_file(DUAS_GEOJSON_PATH)
        else:
            print("📍 Creating 2-class reference (no DUA polygons)")
            duas_gdf = None

        # Create reference labels from the GHSL file we just downloaded
        # ✅ NO REDUNDANT DOWNLOAD - Uses local file from Step 1
        reference_filepath = ghsl_downloader.create_reference_labels_from_local(
            local_ghsl_path=ghsl_filepath,
            duas_gdf=duas_gdf,
            output_filename=os.path.join(OUTPUT_DIR, REFERENCE_FILENAME)
        )

        # Show file info
        ref_info = get_file_info(reference_filepath)
        print("\n📋 Reference Labels Information:")
        for key, value in ref_info.items():
            print(f"   {key}: {value}")

        print(f"\n✅ Reference labels created: {reference_filepath}")
    else:
        print("\n⏭️ STEP 2: Skipping reference labels (CREATE_REFERENCE_LABELS = False)")

    # ========================================================================
    # SUMMARY
    # ========================================================================
    print("\n" + "="*50)
    print("🎉 PROCESSING COMPLETE!")
    print("="*50)
    print(f"📂 GHSL Binary Mask: {ghsl_filepath}")
    if reference_filepath:
        print(f"📂 Reference Labels: {reference_filepath}")
    print("="*50)

except Exception as e:
    print(f"\n❌ Processing failed: {str(e)}")
    print(f"   Error type: {type(e).__name__}")
    import traceback
    print(f"   Traceback:\n{traceback.format_exc()}")

    # Set outputs to None on failure
    if 'ghsl_filepath' not in locals():
        ghsl_filepath = None
    if 'reference_filepath' not in locals():
        reference_filepath = None

🏗️ Starting GHSL download and reference creation...

📡 STEP 1: Downloading GHSL built-up data...
--------------------------------------------------
📡 Downloading GHSL built-up data...
   📅 Year: 2020
   🏗️ Threshold: >15 (same as create_ref.py)
   📏 Native scale: 100m
   🔄 Resample to 10m: YES
📊 Loading GHSL dataset: JRC/GHSL/P2023A/GHS_BUILT_S
🔍 Loading GHSL 2020 dataset...
✅ Method 1 successful - Direct image access
🔧 Creating binary built-up mask (replicating create_ref.py)...
📏 Resampling from 100m to 10m...
💾 Downloading to Colab...
📊 Estimated size 316.9 MB > 40MB limit
🔄 Switching to tiled download...
🗂️ Creating tiles (size: 0.1° = ~11.1 km)...
📦 Created 146 tiles for download
📥 Downloading tile 1/146...
🌐 Direct download attempt 1/3...
📦 Processing download...
📥 Downloading tile 2/146...
🌐 Direct download attempt 1/3...
📦 Processing download...
📥 Downloading tile 3/146...
🌐 Direct download attempt 1/3...
📦 Processing download...
📥 Downloading tile 4/146...
🌐 Direct download at

In [ ]:
# ============================================================================
# CELL 7: Download GHSL Data
# ============================================================================

print("🏗️ Starting GHSL download...")
print("="*50)

try:
    # Initialize GHSL downloader
    ghsl_downloader = GHSLDownloader(validate_outputs=VALIDATE_OUTPUTS)

    # Download GHSL data
    ghsl_filepath = ghsl_downloader.download_ghsl_builtup(
        aoi_geom=aoi_geom,
        year=GHSL_YEAR,
        threshold=GHSL_THRESHOLD,
        resample_to_s2=GHSL_RESAMPLE_TO_S2,
        target_scale=S2_SCALE,
        output_filename=os.path.join(OUTPUT_DIR, GHSL_FILENAME)
    )

    # Get download statistics
    ghsl_stats = ghsl_downloader.get_download_stats()

    print("\n📊 GHSL Download Statistics:")
    for key, value in ghsl_stats.items():
        print(f"   {key}: {value}")

    # Show file info
    ghsl_info = get_file_info(ghsl_filepath)
    print("\n📋 GHSL File Information:")
    for key, value in ghsl_info.items():
        print(f"   {key}: {value}")

    print(f"\n✅ GHSL download completed: {ghsl_filepath}")

except Exception as e:
    print(f"❌ GHSL download failed: {str(e)}")
    ghsl_filepath = None

# ============================================================================
# CELL 8: Create Reference Labels (Optional)
# ============================================================================

reference_filepath = None

if CREATE_REFERENCE_LABELS:
    print("🏷️ Creating reference labels...")
    print("="*50)

    try:
        # Create reference labels
        if DUAS_GEOJSON_PATH:
            print(f"📍 Including DUA polygons from: {DUAS_GEOJSON_PATH}")
            import geopandas as gpd
            duas_gdf = gpd.read_file(DUAS_GEOJSON_PATH)
        else:
            print("📍 Creating 2-class reference (no DUA polygons)")
            duas_gdf = None

        reference_filepath = ghsl_downloader.create_reference_labels(
            aoi_geom=aoi_geom,
            year=GHSL_YEAR,
            threshold=GHSL_THRESHOLD,
            duas_gdf=duas_gdf,
            target_scale=S2_SCALE,
            output_filename=os.path.join(OUTPUT_DIR, REFERENCE_FILENAME)
        )

        # Show file info
        ref_info = get_file_info(reference_filepath)
        print("\n📋 Reference Labels Information:")
        for key, value in ref_info.items():
            print(f"   {key}: {value}")

        print(f"\n✅ Reference labels created: {reference_filepath}")

    except Exception as e:
        print(f"❌ Reference labels creation failed: {str(e)}")
        reference_filepath = None
else:
    print("⏭️ Skipping reference labels creation (CREATE_REFERENCE_LABELS = False)")

# ============================================================================
# CELL 9: Validate Data Compatibility
# ============================================================================

if VALIDATE_OUTPUTS and s2_filepath:
    print("🔍 Performing comprehensive data validation...")
    print("="*60)

    try:
        # Initialize validator
        validator = MLCompatibilityValidator()

        # Perform complete dataset validation
        validation_results = validator.validate_complete_dataset(
            s2_file=s2_filepath,
            ghsl_file=ghsl_filepath,
            reference_file=reference_filepath,
            is_s2_normalized=S2_NORMALIZE,
            has_duas=(DUAS_GEOJSON_PATH is not None)
        )

        # Check if ready for ML inference
        if validation_results['dataset_valid']:
            print("\n🎯 DATASET READY FOR ML INFERENCE! 🎯")
        else:
            print("\n❌ Dataset validation failed. Check errors above.")

        # Generate validation report
        report_path = os.path.join(OUTPUT_DIR, "validation_report.txt")
        validator.generate_validation_report(report_path)

        # Save validation results to JSON
        results_path = os.path.join(OUTPUT_DIR, "validation_results.json")
        validator.save_validation_results(validation_results, results_path)

        # Create visualization plots (if matplotlib available)
        try:
            validator.create_validation_plots(validation_results)
        except Exception as e:
            print(f"⚠️ Could not create plots: {str(e)}")

    except Exception as e:
        print(f"❌ Validation failed: {str(e)}")
else:
    print("⏭️ Skipping validation (VALIDATE_OUTPUTS = False or no S2 data)")

# ============================================================================
# CELL 10: Check Spatial Alignment
# ============================================================================

# Check spatial alignment between downloaded files
files_to_check = []
if s2_filepath:
    files_to_check.append(s2_filepath)
if ghsl_filepath:
    files_to_check.append(ghsl_filepath)
if reference_filepath:
    files_to_check.append(reference_filepath)

if len(files_to_check) > 1:
    print("🔍 Checking spatial alignment...")
    print("="*40)

    # Check alignment between all pairs
    for i in range(len(files_to_check)):
        for j in range(i + 1, len(files_to_check)):
            file1, file2 = files_to_check[i], files_to_check[j]

            print(f"\n📐 Checking: {os.path.basename(file1)} vs {os.path.basename(file2)}")
            is_aligned = check_files_alignment(file1, file2)

            if not is_aligned:
                print("⚠️ Files are not aligned! This may cause issues with ML processing.")
                print("Consider using align_to_reference() function to fix alignment.")
else:
    print("ℹ️ Need at least 2 files for alignment check")

# ============================================================================
# CELL 11: Create Inference Patches (Optional)
# ============================================================================

if CREATE_PATCHES and s2_filepath:
    print("🔧 Creating inference patches...")
    print("="*40)

    try:
        # Create patches directory
        patches_dir = os.path.join(OUTPUT_DIR, "patches")
        os.makedirs(patches_dir, exist_ok=True)

        # Use your existing patch creation functionality
        # This would need to be adapted from your extract_patch.py
        print(f"Creating {PATCH_SIZE}x{PATCH_SIZE} patches...")

        # Basic patch creation example (adapt as needed)
        import rasterio
        from rasterio.windows import Window

        with rasterio.open(s2_filepath) as src:
            height, width = src.shape
            patch_count = 0

            for y in range(0, height - PATCH_SIZE + 1, PATCH_SIZE):
                for x in range(0, width - PATCH_SIZE + 1, PATCH_SIZE):
                    # Read patch
                    window = Window(x, y, PATCH_SIZE, PATCH_SIZE)
                    patch_data = src.read(window=window)

                    # Save patch
                    patch_filename = f"patch_{patch_count:04d}.tif"
                    patch_path = os.path.join(patches_dir, patch_filename)

                    # Create output profile
                    profile = src.profile.copy()
                    profile.update({
                        'height': PATCH_SIZE,
                        'width': PATCH_SIZE,
                        'transform': rasterio.windows.transform(window, src.transform)
                    })

                    with rasterio.open(patch_path, 'w', **profile) as dst:
                        dst.write(patch_data)

                    patch_count += 1

        print(f"✅ Created {patch_count} patches in {patches_dir}")

    except Exception as e:
        print(f"❌ Patch creation failed: {str(e)}")
else:
    print("⏭️ Skipping patch creation (CREATE_PATCHES = False)")

# ============================================================================
# CELL 12: Save to Google Drive (Optional)
# ============================================================================

if SAVE_TO_DRIVE:
    print("💾 Copying files to Google Drive...")
    print("="*40)

    try:
        import shutil

        # Create Drive directory
        drive_dir = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
        os.makedirs(drive_dir, exist_ok=True)

        files_copied = 0

        # Copy downloaded files
        if s2_filepath and os.path.exists(s2_filepath):
            shutil.copy2(s2_filepath, drive_dir)
            print(f"📁 Copied: {os.path.basename(s2_filepath)}")
            files_copied += 1

        if ghsl_filepath and os.path.exists(ghsl_filepath):
            shutil.copy2(ghsl_filepath, drive_dir)
            print(f"📁 Copied: {os.path.basename(ghsl_filepath)}")
            files_copied += 1

        if reference_filepath and os.path.exists(reference_filepath):
            shutil.copy2(reference_filepath, drive_dir)
            print(f"📁 Copied: {os.path.basename(reference_filepath)}")
            files_copied += 1

        # Copy validation report if it exists
        report_path = os.path.join(OUTPUT_DIR, "validation_report.txt")
        if os.path.exists(report_path):
            shutil.copy2(report_path, drive_dir)
            print(f"📁 Copied: validation_report.txt")
            files_copied += 1

        # Copy validation results if it exists
        results_path = os.path.join(OUTPUT_DIR, "validation_results.json")
        if os.path.exists(results_path):
            shutil.copy2(results_path, drive_dir)
            print(f"📁 Copied: validation_results.json")
            files_copied += 1

        print(f"\n✅ Copied {files_copied} files to Google Drive folder: {DRIVE_FOLDER}")

    except Exception as e:
        print(f"❌ Failed to copy to Google Drive: {str(e)}")
else:
    print("⏭️ Skipping Google Drive save (SAVE_TO_DRIVE = False)")

# ============================================================================
# CELL 13: Summary and Next Steps
# ============================================================================

# Final summary
print("\n" + "="*60)
print("📊 DOWNLOAD SUMMARY")
print("="*60)

print(f"📅 Processing date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🗺️ AOI: {AOI_BOUNDS if AOI_GEOJSON_PATH is None else AOI_GEOJSON_PATH}")
print(f"📅 Date range: {START_DATE} to {END_DATE}")
print(f"📁 Output directory: {OUTPUT_DIR}")

print("\n📁 Downloaded Files:")
if s2_filepath and os.path.exists(s2_filepath):
    size_mb = os.path.getsize(s2_filepath) / (1024*1024)
    print(f"   ✅ Sentinel-2: {os.path.basename(s2_filepath)} ({size_mb:.1f} MB)")
else:
    print(f"   ❌ Sentinel-2: Download failed")

if ghsl_filepath and os.path.exists(ghsl_filepath):
    size_mb = os.path.getsize(ghsl_filepath) / (1024*1024)
    print(f"   ✅ GHSL: {os.path.basename(ghsl_filepath)} ({size_mb:.1f} MB)")
else:
    print(f"   ❌ GHSL: Download failed")

if reference_filepath and os.path.exists(reference_filepath):
    size_mb = os.path.getsize(reference_filepath) / (1024*1024)
    print(f"   ✅ Reference: {os.path.basename(reference_filepath)} ({size_mb:.1f} MB)")
else:
    print(f"   ⏭️ Reference: Not created")

print("\n🎯 Next Steps:")
if s2_filepath and os.path.exists(s2_filepath):
    print("   1. ✅ Data is ready for ML model inference")
    print("   2. 📤 Load data into your ML pipeline")
    print("   3. 🔧 Create patches if needed for your model")
    print("   4. 🚀 Run inference on downloaded data")
else:
    print("   1. ❌ Fix download issues before proceeding")
    print("   2. 🔍 Check error messages above")
    print("   3. 🔄 Consider adjusting parameters and re-running")

print("\n💡 Tips:")
print("   - Files are exactly compatible with your original preprocessing pipeline")
print("   - S2 data includes harmonization matching your harmonize.py")
print("   - GHSL data uses same threshold (>15) as your create_ref.py")
print("   - All spatial resolutions are aligned at 10m")

print("\n" + "="*60)
print("🎉 PROCESSING COMPLETED!")
print("="*60)

# ============================================================================
# CELL 14: Load Data for ML Processing (Example)
# ============================================================================

# Example: Load downloaded data for ML processing
if s2_filepath and os.path.exists(s2_filepath):
    print("🔧 Example: Loading data for ML processing...")

    import rasterio
    import numpy as np

    # Load S2 data
    with rasterio.open(s2_filepath) as src:
        s2_data = src.read()  # Shape: (bands, height, width)
        s2_profile = src.profile

        print(f"📊 S2 data shape: {s2_data.shape}")
        print(f"📊 S2 data type: {s2_data.dtype}")
        print(f"📊 S2 value range: [{s2_data.min():.3f}, {s2_data.max():.3f}]")

    # Load GHSL data if available
    if ghsl_filepath and os.path.exists(ghsl_filepath):
        with rasterio.open(ghsl_filepath) as src:
            ghsl_data = src.read(1)  # Shape: (height, width)

            print(f"📊 GHSL data shape: {ghsl_data.shape}")
            print(f"📊 GHSL unique values: {np.unique(ghsl_data)}")

    # Load reference labels if available
    if reference_filepath and os.path.exists(reference_filepath):
        with rasterio.open(reference_filepath) as src:
            ref_data = src.read(1)  # Shape: (height, width)

            print(f"📊 Reference data shape: {ref_data.shape}")
            print(f"📊 Reference unique values: {np.unique(ref_data)}")

    print("\n💡 Your data is now ready for ML model inference!")
    print("   - S2 data is normalized to [0,1] range")
    print("   - GHSL data is binary (0=non-built, 1=built-up)")
    print("   - All data is spatially aligned at 10m resolution")
    print("   - Data format exactly matches your original preprocessing pipeline")

else:
    print("⚠️ No S2 data available for ML processing example")

# ============================================================================
# CELL 15: Visualize Downloaded Data (Optional)
# ============================================================================

# Optional: Create simple visualizations of downloaded data
try:
    import matplotlib.pyplot as plt
    import numpy as np

    if s2_filepath and os.path.exists(s2_filepath):
        print("📊 Creating data visualizations...")

        # Load data for visualization
        with rasterio.open(s2_filepath) as src:
            # Read RGB bands (B4, B3, B2 = Red, Green, Blue)
            # Assuming band order: B2, B3, B4, B8, B11, B12
            red = src.read(3)    # B4
            green = src.read(2)  # B3
            blue = src.read(1)   # B2

            # Create RGB composite
            rgb = np.dstack([red, green, blue])

            # Enhance contrast for visualization
            rgb_enhanced = np.clip(rgb * 3, 0, 1)

        # Create visualization
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # RGB composite
        axes[0].imshow(rgb_enhanced)
        axes[0].set_title('Sentinel-2 RGB Composite')
        axes[0].axis('off')

        # GHSL data if available
        if ghsl_filepath and os.path.exists(ghsl_filepath):
            with rasterio.open(ghsl_filepath) as src:
                ghsl_data = src.read(1)

            axes[1].imshow(ghsl_data, cmap='RdYlBu_r')
            axes[1].set_title('GHSL Built-up Areas')
            axes[1].axis('off')
        else:
            axes[1].text(0.5, 0.5, 'GHSL data\nnot available',
                        ha='center', va='center', transform=axes[1].transAxes)
            axes[1].set_title('GHSL Built-up Areas')
            axes[1].axis('off')

        # Reference labels if available
        if reference_filepath and os.path.exists(reference_filepath):
            with rasterio.open(reference_filepath) as src:
                ref_data = src.read(1)

            axes[2].imshow(ref_data, cmap='viridis')
            axes[2].set_title('Reference Labels')
            axes[2].axis('off')
        else:
            axes[2].text(0.5, 0.5, 'Reference labels\nnot available',
                        ha='center', va='center', transform=axes[2].transAxes)
            axes[2].set_title('Reference Labels')
            axes[2].axis('off')

        plt.tight_layout()
        plt.show()

        print("✅ Data visualization completed!")

except ImportError:
    print("⚠️ Matplotlib not available. Skipping visualization.")
except Exception as e:
    print(f"⚠️ Visualization failed: {str(e)}")

# ============================================================================
# CELL 16: Export Data Summary
# ============================================================================

# Create a summary of all downloaded data
summary_data = {
    'processing_info': {
        'processing_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'aoi': AOI_BOUNDS if AOI_GEOJSON_PATH is None else AOI_GEOJSON_PATH,
        'date_range': f"{START_DATE} to {END_DATE}",
        'output_directory': OUTPUT_DIR
    },
    'configuration': {
        's2_bands': S2_BANDS,
        's2_harmonization': S2_APPLY_HARMONIZATION,
        's2_normalization': S2_NORMALIZE,
        'ghsl_year': GHSL_YEAR,
        'ghsl_threshold': GHSL_THRESHOLD,
        'spatial_resolution': f"{S2_SCALE}m"
    },
    'files': {}
}

# Add file information
if s2_filepath and os.path.exists(s2_filepath):
    summary_data['files']['sentinel2'] = {
        'filename': os.path.basename(s2_filepath),
        'size_mb': round(os.path.getsize(s2_filepath) / (1024*1024), 2),
        'status': 'success'
    }
else:
    summary_data['files']['sentinel2'] = {'status': 'failed'}

if ghsl_filepath and os.path.exists(ghsl_filepath):
    summary_data['files']['ghsl'] = {
        'filename': os.path.basename(ghsl_filepath),
        'size_mb': round(os.path.getsize(ghsl_filepath) / (1024*1024), 2),
        'status': 'success'
    }
else:
    summary_data['files']['ghsl'] = {'status': 'failed'}

if reference_filepath and os.path.exists(reference_filepath):
    summary_data['files']['reference'] = {
        'filename': os.path.basename(reference_filepath),
        'size_mb': round(os.path.getsize(reference_filepath) / (1024*1024), 2),
        'status': 'success'
    }
else:
    summary_data['files']['reference'] = {'status': 'not_created'}

# Save summary to JSON
summary_path = os.path.join(OUTPUT_DIR, "download_summary.json")
try:
    import json
    with open(summary_path, 'w') as f:
        json.dump(summary_data, f, indent=2)
    print(f"📄 Download summary saved to: {summary_path}")
except Exception as e:
    print(f"⚠️ Could not save summary: {str(e)}")

# Print final status
print("\n" + "="*60)
print("🎯 FINAL STATUS")
print("="*60)

total_files = len([f for f in [s2_filepath, ghsl_filepath, reference_filepath] if f and os.path.exists(f)])
expected_files = 2 + (1 if CREATE_REFERENCE_LABELS else 0)  # S2 + GHSL + optional reference

success_rate = (total_files / expected_files) * 100 if expected_files > 0 else 0

print(f"📊 Success rate: {success_rate:.1f}% ({total_files}/{expected_files} files)")

if success_rate == 100:
    print("🎉 ALL DOWNLOADS COMPLETED SUCCESSFULLY!")
    print("✅ Your data is ready for ML model inference")
elif success_rate >= 50:
    print("⚠️ PARTIAL SUCCESS - Some files downloaded")
    print("🔍 Check individual download results above")
else:
    print("❌ DOWNLOAD FAILED - Most files could not be downloaded")
    print("🔧 Review configuration and try again")

print("\n🔗 Related files:")
print(f"   📄 Validation report: validation_report.txt")
print(f"   📊 Validation results: validation_results.json")
print(f"   📋 Download summary: download_summary.json")

if SAVE_TO_DRIVE:
    print(f"\n💾 Files also saved to Google Drive: {DRIVE_FOLDER}")

print("\n🚀 Ready for ML inference! Load the data and run your models.")
print("="*60)